# 🚦 RoadSign Evaluator — Entrenamiento AVANZADO de Modelo IA

Este cuaderno **entrena de verdad** un modelo de detección de señales de tráfico europeas/españolas usando YOLOv8 sobre miles de imágenes reales etiquetadas, y lo exporta listo para tu app.

**Diferencia con el cuaderno básico:** aquel solo convertía un modelo genérico. Este **entrena uno especializado** en señales europeas con data augmentation, lo que da mucha mayor precisión real.

---

## ⚙️ ANTES DE EMPEZAR — Activa la GPU gratis (importante)

1. Menú: **Entorno de ejecución → Cambiar tipo de entorno de ejecución**
2. En **Acelerador por hardware** elige **GPU T4**
3. Guardar

Sin esto el entrenamiento tardaría días en vez de 1-2 horas.

## 🔑 Necesitas una API key GRATIS de Roboflow (para descargar el dataset)

1. Crea cuenta gratis en https://roboflow.com
2. Ve a tu perfil → **Settings → API Keys** → copia tu **Private API Key**
3. Pégala en la celda del Paso 2 donde pone `TU_API_KEY`

---

Luego: **Entorno de ejecución → Ejecutar todo** y espera. Al final se descarga `model.onnx` + `labels.json`.

## Paso 1 — Verificar GPU e instalar herramientas

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'Tesla' in gpu.stdout or 'T4' in gpu.stdout or 'GPU' in gpu.stdout:
    print('✅ GPU activa:')
    print(gpu.stdout.split('\n')[8] if len(gpu.stdout.split('\n'))>8 else 'GPU detectada')
else:
    print('⚠️ NO HAY GPU. Ve a: Entorno de ejecución → Cambiar tipo → GPU T4')
    print('   Puedes continuar pero será MUY lento.')

%pip install -q ultralytics roboflow onnx onnxslim
print('\n✅ Herramientas instaladas')

## Paso 2 — Descargar el dataset de señales europeas

Usamos **Traffic Signs Detection Europe** (4.381 imágenes, 55 clases de señales europeas), el más alineado con el catálogo español. Pega tu API key de Roboflow abajo.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PEGA AQUÍ TU API KEY DE ROBOFLOW (gratis en roboflow.com)
TU_API_KEY = "TU_API_KEY"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from roboflow import Roboflow
rf = Roboflow(api_key=TU_API_KEY)

# Dataset europeo principal (señales tipo España/Convención de Viena)
project = rf.workspace("radu-oprea-r4xnm").project("traffic-signs-detection-europe")
dataset = project.version(14).download("yolov8")

print(f'\n✅ Dataset descargado en: {dataset.location}')

# Localizar el data.yaml
import os, glob
yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f'Config: {yaml_path}')
with open(yaml_path) as f:
    print('\n--- data.yaml ---')
    print(f.read())

## Paso 3 — Entrenar el modelo (lo más preciso posible)

Usamos **YOLOv8s** (small) con data augmentation agresivo y 150 épocas con parada temprana. En GPU T4 tarda aproximadamente 1-2 horas.

Si prefieres un modelo más ligero y rápido (menos preciso), cambia `yolov8s.pt` por `yolov8n.pt`.

In [ ]:
from ultralytics import YOLO

# Modelo base pre-entrenado (transfer learning)
model = YOLO('yolov8s.pt')   # 's' = small (preciso). Cambia a 'yolov8n.pt' para más velocidad

results = model.train(
    data=yaml_path,
    epochs=150,             # vueltas al dataset
    patience=25,            # parar si no mejora en 25 épocas (early stopping)
    imgsz=640,
    batch=16,               # baja a 8 si da error de memoria GPU
    
    # Data augmentation agresivo para robustez en condiciones reales
    hsv_h=0.015,            # variación de tono
    hsv_s=0.7,              # variación de saturación
    hsv_v=0.4,              # variación de brillo (luz/sombra)
    degrees=10,             # rotación ±10°
    translate=0.1,          # traslación
    scale=0.5,              # zoom
    fliplr=0.0,             # NO voltear horizontal (las señales tienen orientación)
    mosaic=1.0,             # mosaico (combina 4 imágenes)
    mixup=0.15,             # mezcla de imágenes
    
    optimizer='auto',
    lr0=0.01,
    cos_lr=True,            # learning rate coseno (mejor convergencia)
    
    project='roadsign_train',
    name='europe_signs',
    exist_ok=True,
    verbose=True,
)
print('\n✅ Entrenamiento completado')
print(f'Mejor modelo: roadsign_train/europe_signs/weights/best.pt')

## Paso 4 — Ver métricas de precisión

mAP50 es la métrica clave: cuanto más cerca de 1.0, mejor detecta. Por encima de 0.7 es un buen modelo.

In [ ]:
best = YOLO('roadsign_train/europe_signs/weights/best.pt')
metrics = best.val(data=yaml_path)
print(f'\n📊 RESULTADOS:')
print(f'  mAP50:    {metrics.box.map50:.3f}  (precisión a IoU 0.5)')
print(f'  mAP50-95: {metrics.box.map:.3f}  (precisión media estricta)')
print(f'  Precisión: {metrics.box.mp:.3f}')
print(f'  Recall:    {metrics.box.mr:.3f}')

# Mostrar curva de entrenamiento
from IPython.display import Image, display
import os
results_png = 'roadsign_train/europe_signs/results.png'
if os.path.exists(results_png):
    display(Image(results_png))

## Paso 5 — Exportar a ONNX para la app

In [ ]:
import shutil, json, os

# Exportar el MEJOR modelo a ONNX
onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.move(onnx_path, 'model.onnx')
print('✅ model.onnx generado')

# Extraer etiquetas en el orden correcto
names = best.names
labels = [names[i] for i in range(len(names))]
with open('labels.json', 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)
print(f'✅ labels.json con {len(labels)} clases')

size_mb = os.path.getsize('model.onnx')/1024/1024
print(f'\n📦 Tamaño: {size_mb:.1f} MB')
print('\nClases detectables:')
for i, n in enumerate(labels):
    print(f'  {i}: {n}')

## Paso 6 — Generar el mapeo de clases para tu app

Este paso crea automáticamente un archivo `classMapper_generado.txt` que te dice cómo conectar las clases del modelo con tu catálogo DGT. Cópialo en `js/detection/classMapper.js` si las clases no coinciden con el GTSRB estándar.

In [ ]:
# Generar sugerencia de mapeo basada en los nombres de clase
mapping_lines = ['// Mapa generado automáticamente — revisa y ajusta el signType según tu catálogo DGT', 'const GENERATED_MAP = {']
for i, name in enumerate(labels):
    n = name.lower()
    if 'forbidd' in n or 'prohib' in n or 'no-' in n:
        cat, st = 'prohibicion', 'R100'
    elif 'warning' in n or 'danger' in n:
        cat, st = 'peligro', 'P18'
    elif 'mandator' in n or 'oblig' in n:
        cat, st = 'obligacion', 'M501'
    elif 'inform' in n:
        cat, st = 'informacion', 'S10'
    elif 'stop' in n:
        cat, st = 'prioridad', 'R2'
    elif 'yield' in n or 'give-way' in n:
        cat, st = 'prioridad', 'R1'
    elif 'speed' in n or 'limit' in n:
        cat, st = 'velocidad', 'R203'
    else:
        cat, st = 'desconocido', 'UNKNOWN'
    mapping_lines.append(f"  {i}: {{ signType:'{st}', category:'{cat}' }},  // {name}")
mapping_lines.append('};')
mapping = '\n'.join(mapping_lines)
with open('classMapper_generado.txt', 'w', encoding='utf-8') as f:
    f.write(mapping)
print('✅ classMapper_generado.txt creado')
print('\n(Vista previa — revisa los signType antes de usar)')
print(mapping[:1500])

## Paso 7 — Descargar todo

Se descargan `model.onnx`, `labels.json` y `classMapper_generado.txt`.

**Sube `model.onnx` y `labels.json` a la carpeta `models/` de tu repositorio.**
Si las clases no coinciden con el GTSRB estándar, usa `classMapper_generado.txt` para actualizar `js/detection/classMapper.js`.

In [ ]:
from google.colab import files
files.download('model.onnx')
files.download('labels.json')
files.download('classMapper_generado.txt')
print('\n✅ ¡Listo! Sube model.onnx y labels.json a la carpeta models/ de tu repo.')